## Máxima Resolución, Saturación Semántica y Abstracción (K=15)

Los objetivos de este cuaderno son: evaluar si un modelo de alta fragmentación ($k=15$) es capaz de aislar definitivamente las dimensiones críticas de riesgo para el paciente (conducción y salud reproductiva) que permanecían ocultas en modelos anteriores y sintetizar los resultados en una primera propuesta de "Preguntas Maestras".

Para ello seguimos el mismo procedimiento que para los 11 y 13 clusters.

## Cálculo de embeddings y clustering

In [1]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_limpio_segmentado.csv")

In [2]:
df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

### Hipótesis de Aislamiento de Riesgos Críticos

Tras las iteraciones con $k=11$ y $k=13$, se determinó que temas de extrema importancia para la seguridad del paciente, como la afectación a la capacidad de conducir o el uso durante el embarazo, seguían diluidos. Al elevar el número de clústeres a $k=15$, forzamos al algoritmo a buscar micro-patrones en el espacio vectorial para otorgar a estas advertencias su propia categoría independiente.

In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 15 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

c:\Users\04jul\Desktop\CDIA\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 306.96it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 969/969 [16:46<00:00,  1.04s/it] 


In [4]:
df_parrafos.to_csv("prospectos_clusters15.csv", index=False)

## Preparación de prompts

In [ ]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      
  
  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿El paracetamol afecta a la capacidad de conducción?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me lo tengo que tomar?".
  3. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Genera la PREGUNTA DIRECTA que un paciente normal (no experto) haría. 
  Ejemplos de estilo correcto: 
  - "¿Qué hago si se me olvida una dosis?" 
  - "¿Puedo conducir después de tomar esto?"
  - "¿Dónde debo guardar la medicina?"

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [6]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=150) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

In [7]:
import requests

def listar_modelos_disponibles():
    url_base = "https://wiig.dia.fi.upm.es/ollama/api/tags"
    try:
        response = requests.get(url_base, timeout=10)
        if response.status_code == 200:
            modelos = [m['name'] for m in response.json().get('models', [])]
            print(f" Modelos disponibles: {modelos}")
            return modelos
        else:
            print(f" No se pudo listar modelos. Status: {response.status_code}")
    except Exception as e:
        print(f" Error de conexión: {e}")
    return []

modelos_reales = listar_modelos_disponibles()

 Modelos disponibles: ['qwen3:1.7b', 'qwen3:8b', 'llama3_hard:latest', 'llama3_medium_Open:latest', 'llama3_easy:latest', 'llama3_easy_Open:latest', 'llama3_hard_Open:latest', 'llama3_medium:latest', 'llama3:latest']


In [8]:
anotadores = ["llama3:latest", "llama3_hard:latest", "llama3_medium:latest"]
resultados = []

In [9]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    fila = {}
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_variados)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_variados)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_15topicos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando llama3:latest...
[AIDE]

{
"pregunta_del_paciente": "¿Qué pasa si me olvido tomar [MEDICAMENTO] durante un tiempo? ¿Puedo seguir tomando el medicamento después de un retraso o debo esperar a que termine la próxima dosis?"
}
[/AIDE]
  -> Consultando llama3_hard:latest...
{ "pregunta_del_paciente": "¿Qué pasa si siento dolor pélvico y cambio en el tejido mamario?" }
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Qué hago si siento dolor pélvico mientras tomo [MEDICAMENTO]?"
}
 Procesando Cluster 1...
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Qué hago en caso de olvidar una dosis o tener alguna duda adicional sobre el uso de este medicamento?"
}
  -> Consultando llama3_hard:latest...
{ "pregunta_del_paciente": "¿Puedo dar mi medicamento a alguien más o si eso puede hacer daño?" }
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Puedo dar mi medicamento a mis hijos si se me olvida una

### Evaluación de Resultados: Victorias Clínicas y Consolidación
El escalado a 15 clústeres confirma la hipótesis de aislamiento, logrando rescatar los temas que estaban "escondidos":
* **Cluster 6 (Conducción):** Llama3 Latest y Hard coinciden plenamente (*"¿Puedo conducir o manejar maquinaria después de tomar esto?"*). Es una dimensión vital para la seguridad vial.
* **Cluster 13 (Salud Reproductiva):** Se centra exclusivamente en el embarazo y el deseo de concebir (*"¿Puedo seguir tomando el medicamento si estoy embarazada?"*).

Basándonos en la convergencia de estos 15 clústeres, logramos consolidar el conocimiento en una primera propuesta de **9 Preguntas Maestras**, ya que el resto de clusters son demasiado específicos (abarcando desde la utilidad del fármaco hasta las urgencias y la conservación).

In [ ]:
import pandas as pd
import json
import re

def extraer_pregunta_segura(texto):
    """
    Limpia y extrae la pregunta del JSON generado por el LLM, 
    gestionando texto basura y fallos de formato.
    """
    if pd.isna(texto) or "ERROR" in str(texto):
        return "Error en respuesta"

    match = re.search(r'\{.*\}', str(texto), re.DOTALL)
    
    if match:
        json_str = match.group()
        try:
            # Reemplazar comillas dobles repetidas
            json_str = json_str.replace('""', '"')
            datos = json.loads(json_str)
            
            pregunta = datos.get('pregunta_final')
            
            if pregunta:
                return pregunta.strip()
        except Exception:
            pass

    # Si falla el JSON, buscamos patrones de texto
    # Busca frases que empiecen por "¿" y terminen por "?"
    preguntas_encontradas = re.findall(r'¿[^?]+\?', str(texto))
    if preguntas_encontradas:
        # Devolvemos la última pregunta encontrada
        return preguntas_encontradas[-1].strip()

    return "No se pudo identificar la pregunta"


df_resultados = pd.read_csv('resultados_etiquetado_15topicos.csv')

anotadores = [col for col in df_resultados.columns if col.startswith('res_')]

for col in anotadores:
    nombre_limpio = col.replace('res_', 'pregunta_')
    df_resultados[nombre_limpio] = df_resultados[col].apply(extraer_pregunta_segura)

columnas_formulario = ['cluster'] + [col for col in df_resultados.columns if col.startswith('pregunta_')]
print(df_resultados[columnas_formulario].head())

df_resultados[columnas_formulario].to_csv('preguntas_para_formulario_15.csv', index=False)

   cluster                             pregunta_llama3_latest  \
0        0  ¿Puedo seguir tomando el medicamento después d...   
1        1  ¿Qué hago en caso de olvidar una dosis o tener...   
2        2  ¿debo seguir tomándolos o hablar con mi médico...   
3        3          ¿Qué contiene esta lista de medicamentos?   
4        4    ¿Cuál es la dosis correcta para mi edad y peso?   

                         pregunta_llama3_hard_latest  \
0  ¿Qué pasa si siento dolor pélvico y cambio en ...   
1  ¿Puedo dar mi medicamento a alguien más o si e...   
2  ¿Debo seguir tomando medicina o consultar con ...   
3  ¿Qué hago si estoy alérgico a la soja y toco [...   
4  ¿Qué debo hacer si accidentalmente carga una d...   

                       pregunta_llama3_medium_latest  
0  ¿Qué hago si siento dolor pélvico mientras tom...  
1  ¿Puedo dar mi medicamento a mis hijos si se me...  
2  ¿Qué hago si tengo alguna dolencia o estoy tom...  
3  ¿Puedo seguir tomando esta medicina si tengo a...

### Limitaciones

Aunque la Abstracción Semántica Post-Clustering logró universalizar las preguntas (ej. transformando "ojos/recto" en "¿Existen instrucciones especiales según la zona de aplicación?"), el modelo de 15 clústeres sacó a la luz un problema de fondo insalvable mediante *Prompt Engineering*: **La Redundancia Estructural del texto original.**

El tema del "Olvido de dosis" aparece replicado en 5 clústeres distintos (1, 5, 6, 8, 14). Esto no es un fallo del algoritmo K-Means, sino una evidencia métrica de la fragmentación de la AEMPS, que repite esta instrucción en casi todas sus secciones legales (Introducción, Instrucciones, Información Adicional).

**Siguiente Paso Metodológico:** Se concluye que hemos alcanzado el límite de la "Saturación Semántica". El aumento del volumen de datos (500 prospectos) sin filtrado previo refuerza el ruido en lugar de aportar diversidad. Para lograr una arquitectura de información óptima, la siguiente fase abandonará el ajuste de clústeres y se centrará en intervenir la fuente de datos mediante una **Deduplicación Semántica (Filtro TF-IDF)** antes de volver a vectorizar.